In [1]:
import json
import os
from tqdm import tqdm

In [2]:
def load_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)

def save_json(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as file:
        json.dump(data, file, ensure_ascii=False, indent=4)

In [ ]:
import re

def get_number_from_string(text):
    # Tìm dấu gạch dưới và các chữ số ở cuối chuỗi
    match = re.search(r'_(\d+)$', text)
    if match:
        return int(match.group(1)) # Trả về số dưới dạng string
    return None



[10, 25, 99]


In [4]:
CACHE_URL_PATH = r"data\cache\mapping\hash2url.json"
CACHE_SEARCH_PATH = r"data\cache\mapping\hash_id_IR_20.json"
METADATA_PATH = r"data\metadata_9000q.json" #1000
# METADATA_PATH = r"data\metadata_medhop.json" #1000

DIR_SEARCH = "data/cache/IR_20"
DIR_WIKI = "data/cache/wiki_pages_tool_1"

In [5]:
def flatten_wiki_cache(file_path):
    # 1. Load dữ liệu từ file
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
    except Exception as e:
        print(f"Lỗi khi đọc file: {e}")
        return {}

    final_cache = {}


    for main_id, tasks in raw_data.items():
        for task_name, items in tasks.items():
            for hash_id, url in items.items():
                final_cache[hash_id] = url

    return final_cache

FILE_PATH = r'data\cache\all_hash_id_crawl_wiki_embed.json' 
url_mapping = flatten_wiki_cache(FILE_PATH)

print(f"Tổng số HashID thu được: {len(url_mapping)}")

Lỗi khi đọc file: [Errno 2] No such file or directory: 'data\\cache\\all_hash_id_crawl_wiki_embed.json'
Tổng số HashID thu được: 0


In [6]:
CACHE_URL_WIKI = load_json(CACHE_URL_PATH)
CACHE_SEARCH = load_json(CACHE_SEARCH_PATH)
metadata = load_json(METADATA_PATH)
    
CACHE_URL_WIKI.update(url_mapping)

# SEARCH_MAPPING = load_json(r'data\cache\mapping\hash_id copy.json')
# CACHE_SEARCH.update(SEARCH_MAPPING)
# CACHE_SEARCH.update(load_json(r'data\cache\mapping\hash_id.json'))

# save_json(CACHE_SEARCH, CACHE_SEARCH_PATH)

In [7]:
len(CACHE_URL_WIKI)

53017

In [8]:
import re

def load_retrieval_map(data_input):
    """
    Map ID (WIKI_10, CRAWL_NER_WIKI_2,...) trực tiếp với hash_id theo thứ tự 1-1.
    """
    result_map = {}
    
    for key, entry in data_input.items():
        crawl_docs = entry.get("crawl_doc", [])
        hash_ids = entry.get("hash_id", [])
        
        for i, doc_text in enumerate(crawl_docs):
            # Regex mới: 
            # \[ : Bắt đầu ngoặc vuông
            # ( : Bắt đầu group muốn lấy
            #  .*? : Mọi ký tự (WIKI, CRAWL_NER_WIKI,...)
            #  _ : Dấu gạch dưới
            #  \d+ : Các chữ số
            # ) : Kết thúc group
            # \] : Đóng ngoặc vuông
            match = re.search(r'\[(.*?_\d+)\]', doc_text)
            
            if match and i < len(hash_ids):
                source_key = match.group(1) 
                result_map[source_key] = hash_ids[i]
                
    return result_map

In [9]:
def extract_detail_regex(text):
    pattern = r'(?i)Detailed explanation[:\s]*(.*)'
    match = re.search(pattern, text)
    
    if match:
        return match.group(1).strip()
    return ""

# DIRECT

In [10]:
BASE_9000 = "DIRECT_9000_medhop_results/all_ans"
DIR_SEARCH = "data/cache/IR_20"
DIR_WIKI = "data/cache/wiki_pages_tool_1"

In [11]:
output_direct = {}
for qidx, item in metadata.items():
    if item['type_q'] != 'direct':
        continue
    
    if qidx != "851819":
        continue
    
    file_path = f"{BASE_9000}/R4_step_3b_all_ans_{qidx}.json"
    tmp_data = load_json(file_path)
    
    final_answer = tmp_data['1']['0']['0']['final_answer']
    question = tmp_data['1']['0']['0']['question']
    search_kw = tmp_data['1']['0']['0']['search_keyword'].lower().strip()
    search_kw_raw = search_kw + " "
    short_answer = final_answer['short_answer']
    long_answer = final_answer['long_answer']
    
    reference_inf = f"Question: {question}\n Answer: {short_answer}\n Reference:\n "
    seen_links = set()  
    source_to_url = {}
    
    if final_answer['selection_status'] != 'selected':
        reference_inf += "Zeroshot - O3" + " \n "
        detail_explanation = long_answer
    else:
        detail_explanation = extract_detail_regex(long_answer)
        sources = final_answer['sources']
        retrieval_information = tmp_data['1']['0']['0']['retrieval_information']
        
        if search_kw in CACHE_SEARCH or search_kw_raw in CACHE_SEARCH:
            hash_id_gg = CACHE_SEARCH.get(search_kw) or CACHE_SEARCH.get(search_kw_raw)
            if '.json' not in hash_id_gg:
                hash_id_gg = f"{hash_id_gg}.json"
            file_gg = f'{DIR_SEARCH}/{hash_id_gg}'
            gg_data = load_json(file_gg)['google']['search_results']
        else:
            print(f"CACHE 0 co SEARCH GG  {qidx}")
            gg_data = None
        
        crawl_wiki_data = load_retrieval_map(retrieval_information)
        
        for src in sources:
            index_link = get_number_from_string(src)
            if index_link:
                index_link = index_link - 1 # start index = 0
            else:            
                continue
            
            current_link = ""
    
            if 'GG' in src and gg_data:            
                if 0 <= index_link < len(gg_data):
                    current_link = gg_data[index_link]['link']
                else:
                    print(f"Warning: Index {index_link+1} ngoài phạm vi của gg_data (len={len(gg_data)}) tại QIDX: {qidx}")
                    continue 
            elif 'GG' not in src:
                if src not in crawl_wiki_data:
                    continue
                wiki_hashid = crawl_wiki_data[src]
                current_link = CACHE_URL_WIKI.get(wiki_hashid, "")

            if current_link and current_link not in seen_links:
                reference_inf += f"[{src}] " + current_link + "\n "
                seen_links.add(current_link)
                source_to_url[f"[{src}]"] = current_link
                        
        # # --- BƯỚC QUAN TRỌNG: CHÈN LINK VÀO DETAIL ---
        # sorted_tags = sorted(source_to_url.keys(), key=len, reverse=True)
        
        # for tag in sorted_tags:
        #     url = source_to_url[tag]
        #     # Thay thế [GG_1] bằng [GG_1] (http://...)
        #     detail_explanation = detail_explanation.replace(tag, f"{tag} ({url})")
        
    reference_inf += f"Detailed explanation: {detail_explanation}"
    
    print(reference_inf)
    output_direct[qidx] = {
        'Question': question,
        'short_answer': short_answer,
        'long_answer': long_answer,
        'trace': reference_inf
    }

save_json(output_direct, 'data/EVIDENCE/reference_direct_9000.json')
print(len(output_direct.keys()))

Question: What is the most common cause of the syndrome characterized by inadequate reabsorption in the proximal renal tubules in children?
 Answer: cystinosis
 Reference:
 [CRAWL_WIKI_1] https://en.wikipedia.org/wiki/fanconi_syndrome
 [GG_5] https://pmc.ncbi.nlm.nih.gov/articles/PMC3616759/
 [GG_6] https://academic.oup.com/ndt/article/27/12/4273/1823546
 [GG_10] https://journals.lww.com/sjkd/fulltext/1997/08030/hereditary_and_acquired_renal_tubular_disorders_.5.aspx
 [CRAWL_NER_WIKI_1] https://en.wikipedia.org/wiki/proximal_tubule
 [GG_2] https://en.wikipedia.org/wiki/Fanconi_syndrome
 Detailed explanation: The context clearly states that cystinosis is the most common cause of Fanconi syndrome in children, which is the syndrome characterized by inadequate reabsorption in the proximal renal tubules.Reference: [GG_5], [GG_6]. Detailed explanation: The provided context clearly states that Fanconi syndrome, which is characterized by inadequate reabsorption in the proximal renal tubules in

# SEQUENCE

In [ ]:
output_seq = {}
BASE_SEQ = "SEQUENCE_9000_medhop_results/all_ans"
seq_combine = load_json(r"SEQUENCE_9000_medhop_results\seq_combine[0_9000]_miss_qidx_375532.json")

for qidx, item in metadata.items():
    qidx = str(qidx)
    if item['type_q'] != 'sequence':
        continue
    
    file_path = f"{BASE_SEQ}/R4_step_3b_all_ans_{qidx}.json"
    tmp_data = load_json(file_path)
    
    reference_inf = ""
    prev_answer = ""
    # Khởi tạo set để check trùng link cho toàn bộ các Subquestions của câu hỏi này
    seen_links = set() 

    for step, step_data in tmp_data['1']['0'].items():
        reference_inf += f"Subquestion {int(step)+1}:\n"
        step = str(step)
        
        final_answer = tmp_data['1']['0'][step]['final_answer']
        question = tmp_data['1']['0'][step]['question']
        search_kw = tmp_data['1']['0'][step]['search_keyword']
        
        short_answer = final_answer['short_answer']
        long_answer = final_answer['long_answer']
        
        if step != '0':
            search_kw = search_kw + " " + prev_answer
            
        search_kw = search_kw.lower().strip()
        search_kw_raw = search_kw + " "
        prev_answer = short_answer.lower()
        
        reference_inf += f"Question: {question}\n Answer: {short_answer}\n Reference:\n "
        
        if final_answer['selection_status'] != 'selected':
            reference_inf += "ZEROSHOT - O3\n"
        else:        
            detail_explanation = extract_detail_regex(long_answer)
            sources = final_answer['sources']
            retrieval_information = tmp_data['1']['0'][step]['retrieval_information']
            crawl_wiki_data = load_retrieval_map(retrieval_information)
            
            gg_data = None
            if search_kw in CACHE_SEARCH or search_kw_raw in CACHE_SEARCH:
                hash_id_gg = CACHE_SEARCH.get(search_kw) or CACHE_SEARCH.get(search_kw_raw)
                if '.json' not in hash_id_gg:
                    hash_id_gg = f"{hash_id_gg}.json"
                file_gg = f'{DIR_SEARCH}/{hash_id_gg}'
                gg_data = load_json(file_gg)['google']['search_results']
            
            for src in sources:
                index_link = get_number_from_string(src)
                if not index_link:
                    continue
                index_link -= 1
                
                current_link = ""
                if 'GG' in src and gg_data:            
                    if index_link < len(gg_data):
                        current_link = gg_data[index_link]['link']
                elif 'GG' not in src:
                    if src in crawl_wiki_data:
                        wiki_hashid = crawl_wiki_data[src]
                        current_link = CACHE_URL_WIKI.get(wiki_hashid, "")
                
                if current_link and current_link not in seen_links:
                    reference_inf += f"[{src}] " + current_link + "\n "
                    seen_links.add(current_link)
            
            reference_inf += f"Detailed explanation: {detail_explanation}\n"
        
        reference_inf += "\n" 
        
    reference_inf += "================\nConclusion:\n"
    if qidx in seq_combine:
        reference_inf += f"Answer: {seq_combine[qidx]['short_answer']}\n"
        reference_inf += f"Detailed explanation: {seq_combine[qidx]['long_answer']}"
        output_direct[qidx] = {
            'Question': question,
            'short_answer': short_answer,
            'long_answer': long_answer,
            'trace': reference_inf
        }
        output_seq[qidx] = reference_inf
    else:
        print(f"Missing in seq_combine: {qidx}")

save_json(output_seq, 'data/EVIDENCE/reference_sequence_9000.json')

In [13]:
print(len(output_seq.keys()))

1682


In [14]:
len(output_direct.keys())

1683

# PARALLEL

In [ ]:
output_parallel = {}
BASE_PAR = "PARALLEL_9000_medhop_results/all_ans"

for qidx, item in metadata.items():
    if item['type_q'] != 'parallel':
        continue
    
    file_path = f"{BASE_PAR}/R4_step_3b_all_ans_{qidx}.json"
    tmp_data = load_json(file_path)
    
    reference_inf = ""
    # Tập hợp theo dõi link trùng cho toàn bộ câu hỏi parallel này
    seen_links = set() 
    
    for qi in ["1", "2", "3", "4"]:
        if qi not in tmp_data:
            break
        
        reference_inf += f"Subquestion {qi}:\n"        
        prev_answer = ""
        
        for step, step_data in tmp_data[qi]['0'].items():
            step = str(step)
            final_answer = tmp_data[qi]['0'][step]['final_answer']
            question = tmp_data[qi]['0'][step]['question']
            search_kw = tmp_data[qi]['0'][step]['search_keyword'].lower().strip()
            
            short_answer = final_answer['short_answer']
            long_answer = final_answer['long_answer']
            
            if step != '0':
                search_kw = search_kw + " " + prev_answer
            search_kw_raw = search_kw + " "
            
            reference_inf += f"Question: {question}\n Answer: {short_answer}\n Reference:\n "
            
            if final_answer['selection_status'] != 'selected':
                reference_inf += "ZEROSHOT - O3 \n"
            else:        
                detail_explanation = extract_detail_regex(long_answer)
                sources = final_answer['sources']
                retrieval_information = tmp_data[qi]['0'][step]['retrieval_information']
                crawl_wiki_data = load_retrieval_map(retrieval_information)
                
                gg_data = None
                hash_id_gg = CACHE_SEARCH.get(search_kw) or CACHE_SEARCH.get(search_kw_raw)
                if hash_id_gg:
                    if '.json' not in hash_id_gg:
                        hash_id_gg = f"{hash_id_gg}.json"
                    file_gg = f'{DIR_SEARCH}/{hash_id_gg}'
                    gg_data = load_json(file_gg).get('google', {}).get('search_results', [])
                
                for src in sources:
                    index_link = get_number_from_string(src)
                    if not index_link:
                        continue
                    index_link -= 1
                    
                    current_link = ""
                    if 'GG' in src and gg_data:
                        if index_link < len(gg_data):
                            current_link = gg_data[index_link]['link']
                    elif 'GG' not in src:
                        if src in crawl_wiki_data:
                            wiki_hashid = crawl_wiki_data[src]
                            current_link = CACHE_URL_WIKI.get(wiki_hashid, "")
                    
                    if current_link and current_link not in seen_links:
                        reference_inf += f"[{src}] " + current_link + "\n "
                        seen_links.add(current_link)
                
                reference_inf += f"Detailed explanation: {detail_explanation}\n"
            
            prev_answer = short_answer.lower()
            reference_inf += "\n" 
            
    reference_inf += "================\nConclusion:\n"

    reference_inf += f"Answer: {tmp_data['final_answer']['short_answer']}\n"
    reference_inf += f"Detailed explanation: {tmp_data['final_answer']['long_answer']}"
    output_parallel[qidx] = reference_inf
    output_direct[qidx] = {
        'Question': question,
        'short_answer': short_answer,
        'long_answer': long_answer,
        'trace': reference_inf
    }
   

save_json(output_parallel, 'data/EVIDENCE/reference_parallel_9000.json')

In [16]:
save_json(output_direct, 'data/EVIDENCE/reference_9000q.json')
print(len(output_direct.keys()))

1759


In [17]:
import pandas as pd

data_list = []
for qidx, info in output_direct.items():
    data_list.append({
        'QIDX': qidx,
        'Question': info['Question'],
        'Short Answer': info['short_answer'],
        'Long Answer': info['trace']
    })

df = pd.DataFrame(data_list)

# Xuất ra CSV (sử dụng utf-8-sig để mở bằng Excel không bị lỗi font tiếng Việt)
df.to_csv('data/SUBMIT/submission_9000q.csv', index=False, encoding='utf-8-sig')

print("Đã lưu file CSV thành công!")

Đã lưu file CSV thành công!


In [18]:
import pandas as pd

# Đọc 2 file
df_a = pd.read_csv('data/SUBMIT/submission_1000q.csv')
df_b = pd.read_csv('data/SUBMIT/submission_9000q.csv')

# Nối hàng
df_combined = pd.concat([df_a, df_b], ignore_index=True)

# Lưu lại
df_combined.to_csv('submission.csv', index=False, encoding='utf-8-sig')

In [19]:
len(df_combined)

2745